In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_109_Lodhi_Road_Delhi_IMD_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,137.21,245.44,6.85,11.17,18.02,NaN,NaN,2.22,12.15,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2024-01-02,148.48,260.50,11.33,11.29,22.62,NaN,NaN,2.13,11.97,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2024-01-03,140.68,242.15,15.69,11.39,27.09,NaN,NaN,2.19,17.86,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2024-01-04,177.00,314.73,12.06,11.28,23.34,NaN,NaN,1.61,12.05,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2024-01-05,143.92,276.12,14.82,11.36,24.67,NaN,NaN,1.60,18.91,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,131.40,102.62,29.46,29.90,38.90,NaN,NaN,0.70,5.44,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
362,2024-12-28,62.68,109.24,34.81,14.56,36.05,NaN,NaN,0.52,3.01,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
363,2024-12-29,73.44,127.68,9.62,18.49,17.66,NaN,NaN,0.38,11.23,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
364,2024-12-30,73.73,137.51,13.22,18.14,20.40,NaN,NaN,0.39,14.65,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 10)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['NH3 (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 9)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         137.21        245.44        6.85        11.17   
1  2024-01-02         148.48        260.50       11.33        11.29   
2  2024-01-03         140.68        242.15       15.69        11.39   
3  2024-01-04         177.00        314.73       12.06        11.28   
4  2024-01-05         143.92        276.12       14.82        11.36   

   NOx (ppb)  CO (mg/m³)  Ozone (µg/m³)  Benzene (µg/m³)  
0      18.02        0.63          12.15              0.0  
1      22.62        0.63          11.97              0.0  
2      27.09        0.63          17.86              0.0  
3      23.34        1.61          12.05              0.0  
4      24.67        1.60          18.91              0.0  


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³)
0,2024-01-01,1.650076,1.411148,-0.589381,0.878177,-0.054852,-0.145479,-1.422334,0.0
1,2024-01-02,1.915738,1.616095,-0.122560,0.904583,0.350629,-0.145479,-1.437835,0.0
2,2024-01-03,1.731873,1.366376,0.331756,0.926588,0.744650,-0.145479,-0.930628,0.0
3,2024-01-04,2.588025,2.354094,-0.046493,0.902383,0.414095,2.933001,-1.430946,0.0
4,2024-01-05,1.808247,1.828663,0.241101,0.919987,0.531332,2.901588,-0.840209,0.0
...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.513120,-0.532443,1.766604,-0.142859,1.785676,0.074413,-2.000154,0.0
362,2024-12-28,-0.106780,-0.442353,2.324079,1.624149,1.534454,-0.491022,-2.209409,0.0
363,2024-12-29,0.146860,-0.191409,-0.300744,2.488949,-0.086585,-0.930805,-1.501558,0.0
364,2024-12-30,0.153696,-0.057636,0.074380,2.411931,0.154940,-0.899392,-1.207051,0.0


In [10]:
df.to_excel('LodhiroadIMD2024.xlsx', index=False)